# Ember · 01 · Aggregate FIRMS archives into monthly grid cells

Input: Kaggle dataset `aghetsehngoh/firms-global-hotspots-modis-viirs` (FIRMS per-country yearly CSVs).
Pipeline code: Kaggle dataset `alantchapda/ember-pipeline` (mirror of `pipeline/ember_pipeline` on GitHub).
Output: `hotspots_monthly.parquet`, one row per platform × country × year × month × 0.1° cell (schema in `docs/DATA_CONTRACT.md`).

In [ ]:
import shutil, sys, time, re
from pathlib import Path
import pandas as pd
pkg = Path('/kaggle/working/ember_pipeline'); pkg.mkdir(exist_ok=True)
src = next(p for p in Path('/kaggle/input').rglob('aggregate.py') if 'ember' in str(p)).parent
for f in src.glob('*.py'):
    shutil.copy(f, pkg / f.name)
sys.path.insert(0, '/kaggle/working')
from ember_pipeline.aggregate import aggregate_csv, country_from_name, merge_partials
from ember_pipeline.grid import Grid
print('pipeline loaded from', src)

In [ ]:
RESOLUTION_DEG = 0.1
csvs = sorted(p for p in Path('/kaggle/input').rglob('*.csv') if src not in p.parents)
pattern = re.compile(r'^(?P<sensor>[a-z0-9\-]+)_(?P<year>\d{4})_', re.I)
groups = {}
for p in csvs:
    m = pattern.match(p.name)
    if m:
        groups.setdefault((m['sensor'].lower(), int(m['year'])), []).append(p)
print(len(csvs), 'csv files')
summary = pd.Series({k: len(v) for k, v in groups.items()}).unstack(0)
summary

In [ ]:
out_dir = Path('/kaggle/working/parts'); out_dir.mkdir(exist_ok=True)
grid = Grid(RESOLUTION_DEG)
t0 = time.time()
for (sensor, year), files in sorted(groups.items()):
    target = out_dir / f'{sensor}_{year}.parquet'
    if target.exists():
        continue
    t = time.time()
    part = merge_partials(aggregate_csv(f, grid, country_from_name(f.name)) for f in files)
    part.to_parquet(target, index=False)
    print(f'{sensor} {year}: {len(files)} files -> {len(part):,} rows, {int(part["n"].sum()):,} detections '
          f'({time.time()-t:.0f}s, total {time.time()-t0:.0f}s)', flush=True)

In [ ]:
monthly = pd.concat([pd.read_parquet(p) for p in sorted(out_dir.glob('*.parquet'))], ignore_index=True)
out = Path('/kaggle/working/hotspots_monthly.parquet')
monthly.to_parquet(out, index=False)
shutil.rmtree(out_dir)
print(f'{len(monthly):,} rows -> {out} ({out.stat().st_size/1e6:.1f} MB)')
monthly.groupby('platform')['n'].sum()

## Sanity check: raw yearly detections per platform

In [ ]:
import matplotlib.pyplot as plt
yearly = monthly.groupby(['platform', 'year'])['n'].sum().unstack(0)
ax = yearly.plot(figsize=(10, 4), marker='o')
ax.set_ylabel('detections / year'); ax.set_title('Raw detections per platform, world')
plt.savefig('/kaggle/working/fig_raw_yearly.png', dpi=150, bbox_inches='tight'); plt.show()
yearly